# churn: classification (online, custom-pipeline)

**Use case:** `usecase-type/online/classification/azureml-pipeline` — request/response scoring through an endpoint with the Azure ML pipeline with custom training.

**Training:** ESML pipeline factory `IN_2_GOLD_TRAINING_MANUAL` (medallion lake, data assets, quality-gated evaluation) driven by AzureMLRollout receipts.

**Serving:** Azure ML managed online endpoint via `serving-render`, `serving-deploy` and `serving-invoke`.

**Status:** Executable notebook templates over shared engines; cloud steps are explicit, charged operations behind switches that default to false and need the listed prerequisites.

Kaggle-only source: `blastchar/telco-customer-churn`.
Dataset status: **license-review**.
Blocked until original-author terms and intended use are reviewed; record approval in provenance before changing status to configured. TotalCharges is deliberately excluded because blanks require explicit numeric conversion.

**Prerequisites**

- Review the scenario's dataset status, license and notes; gated datasets stay blocked until approved.
- Install the factory extras `train` (plus `vision` for images, `azure` for Azure ML steps and `streaming` for Event Hubs consumers).
- For Azure steps: `user-config/runtime.local.json` naming an existing workspace, compute, identity and storage selection.
- `user-config/esml-rollout.local.json` with a reviewed source binding for this scenario and pinned Azure ML environments; the ESML SDK is imported from `esml-v2` or azure-esml-sdk.
- Endpoint quota; deployed endpoints bill while running, so delete or scale them to zero when done.

**Limitations**

- Templates are not evidence of a completed Kaggle, Azure or Databricks run.

This notebook is an executable template, not evidence of a completed Kaggle or Azure run.
Install the factory's declared extras into this kernel separately. No packages,
credentials, cloud resources, or competition terms are installed/accepted by this notebook.
All execution switches default to false. Run **either** SDK v2 **or** CLI v2 submission, not both.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "pyproject.toml").is_file()
             and (p / "accelerator" / "src" / "ml_model_factory" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the 50-ml-model-factory checkout")
SOURCE = ROOT / "accelerator" / "src"
sys.path.insert(0, str(SOURCE))
from ml_model_factory.config import load_json, validate_scenario, validate_runtime, write_json

SCENARIO = ROOT / "user-config" / "model" / "scenarios" / "churn.json"
scenario = validate_scenario(load_json(SCENARIO))
MODE = "custom"
PATTERN = "online"
TECHNOLOGY = "azureml-pipeline"
RUNTIME = Path("user-config/runtime.local.json")
if not RUNTIME.is_absolute():
    RUNTIME = ROOT / RUNTIME
WORK = ROOT / "ml-environment" / "outputs" / scenario["name"] / "online-custom-pipeline"
RAW = ROOT / "data" / "in" / scenario["name"]
PREPARED = ROOT / "data" / "out" / scenario["name"] / "online-custom-pipeline" / "prepared"
MODEL = WORK / "model"
EVALUATION = WORK / "evaluation"
RENDERED = WORK / "azureml"
INFERENCE = WORK / PATTERN
REQUESTS = INFERENCE / "requests"
MODEL_NAME = scenario.get("model_name", scenario["name"])
MODEL_ID = ""  # immutable azureml:<name>:<version> after quality-gated registration
RUN_INGEST = False
RUN_PREPARE = False
PREPARE_ESML = False
RUN_ESML_TRAINING = False
SAMPLE_REQUESTS = False
RENDER_ONLINE = False
DEPLOY_ONLINE = False
INVOKE_ONLINE = False
DELETE_ONLINE = False
RUN_ID = None  # defaults to the reviewed binding's run_id; use a new id per attempt
COMPONENT_VERSION = ""  # new pinned pipeline component version, e.g. "1"
SELECTION_POLICY = None  # e.g. ROOT / "user-config" / "model" / "model-selection.json"
NO_CHAMPION = False
ONLINE_ENDPOINT = None
LAKE_CONFIG = ROOT / "user-config" / "lake.local.json"
LAKE_ROOT = ROOT / "data" / "out" / "lake-data"
EFFECTIVE_RUNTIME = RUNTIME
FACTORY_ENV = os.environ.copy()
FACTORY_ENV["PYTHONPATH"] = str(SOURCE) + os.pathsep + FACTORY_ENV.get("PYTHONPATH", "")
FACTORY_ENV.setdefault("MLFLOW_TRACKING_URI", (ROOT / "ml-environment" / "mlruns").as_uri())

def factory(*arguments):
    command = [sys.executable, "-m", "ml_model_factory", *map(str, arguments)]
    subprocess.run(command, cwd=ROOT, env=FACTORY_ENV, check=True)


## Kaggle ingestion and reproducible preparation

Review the source license and any competition rules first. Required-selection and
license-review gates block ingestion. Do not clear them without documented approval.
The download manifest records file hashes and pinned dataset version. Existing nonempty
outputs are never silently replaced. Titanic preparation additionally writes `titanic.parquet`.
Forecasting uses chronological holdouts, not random rows.


In [ ]:
if RUN_INGEST:
    factory("ingest", "--scenario", SCENARIO, "--output", RAW)
INPUT = RAW / scenario["dataset"]["file"]
if RUN_PREPARE:
    factory("prepare", "--scenario", SCENARIO, "--input", INPUT, "--output", PREPARED)
if (PREPARED / "manifest.json").exists():
    print(json.dumps(load_json(PREPARED / "manifest.json"), indent=2))


## ESML pipeline factory: `IN_2_GOLD_TRAINING_MANUAL`

The reusable ESML v2 pipeline factory compiles this scenario's reviewed source binding from
`user-config/esml-rollout.local.json` (schema `esml.azureml-rollout-config/v1`, see the
example file) into a medallion IN-to-BRONZE-to-SILVER-to-GOLD pipeline with a leakage-aware
split, `IN_2_GOLD_TRAINING_MANUAL` and held-out evaluation. `AzureMLRollout` records every cloud write as a
receipt; it never resubmits after an uncertain outcome. Preparation is offline.


In [ ]:
ESML_CONFIG = ROOT / "user-config" / "esml-rollout.local.json"
ESML_OUTPUT = WORK / "esml"


def esml_rollout(cloud=False):
    """Import the ESML SDK (installed or from the purple/orange checkout) and bind this scenario."""
    try:
        import azure_esml  # noqa: F401
    except ModuleNotFoundError:
        for candidate in (ROOT.parents[1] / "esml-v2", ROOT.parents[1] / "azure-enterprise-scale-ml" / "esml-v2"):
            if (candidate / "azure_esml" / "__init__.py").is_file():
                sys.path.insert(0, str(candidate))
                break
        else:
            raise RuntimeError("Install azure-esml-sdk or open this notebook inside the purple/orange checkout")
    from azure_esml import ESMLProject
    from azure_esml.domain_layer.rollout import AzureMLRollout, project_from_rollout_config
    config = load_json(ESML_CONFIG)
    project = project_from_rollout_config(config, ROOT / "user-config" / "model" / "scenarios",
                                          scenario=scenario["name"], mode=MODE, base_path=ESML_CONFIG.parent)
    if cloud:
        from azure_esml.base_layer import AzureMLSDKBackend
        project = ESMLProject(project.settings, backend=AzureMLSDKBackend.from_cli(project.target, subscription_bound=True))
    return AzureMLRollout(project, ESML_OUTPUT), config


if PREPARE_ESML or RUN_ESML_TRAINING:
    rollout, config = esml_rollout(cloud=RUN_ESML_TRAINING)
    binding = config["scenarios"][scenario["name"]]
    RUN_ID = RUN_ID or binding["run_id"]
    from azure_esml.domain_layer.rollout import SCENARIO_NUMBERS
    if PREPARE_ESML:
        request = rollout.prepare(model_number=SCENARIO_NUMBERS[scenario["name"]], mode=MODE,
                                  data_date_utc=config["data_date_utc"], run_id=RUN_ID,
                                  data_version=binding["data_version"])
        print(json.dumps(request, indent=2))
    if RUN_ESML_TRAINING:
        if not COMPONENT_VERSION:
            raise ValueError("Set COMPONENT_VERSION to a new pinned pipeline component version")
        if not (ESML_OUTPUT / RUN_ID / "registered-definitions.json").exists():
            rollout.register_definitions(RUN_ID, component_version=COMPONENT_VERSION)
        policy = load_json(SELECTION_POLICY) if SELECTION_POLICY else None
        print(json.dumps(rollout.execute_training(RUN_ID, selection_policy=policy, no_champion=NO_CHAMPION), indent=2))
if RUN_ID and (ESML_OUTPUT / RUN_ID / "training-completion.json").exists():
    MODEL_ID = load_json(ESML_OUTPUT / RUN_ID / "training-completion.json")["model_id"]
    MODEL_NAME = load_json(ESML_OUTPUT / RUN_ID / "request.json")["model_name"]
    print("MODEL_ID:", MODEL_ID)


## Unlabeled requests from the held-out split

`sample-requests` writes demonstration requests without labels; `online-request.json` is the endpoint payload.
`labels.parquet` keeps held-out truth separately for monitoring demos, and forecasting
also writes `history.parquet` for AutoML scoring. Never send labels to a model.


In [ ]:
if SAMPLE_REQUESTS:
    factory("sample-requests", "--scenario", SCENARIO, "--prepared", PREPARED, "--output", REQUESTS, "--rows", 10)
if (REQUESTS / "manifest.json").exists():
    print(json.dumps(load_json(REQUESTS / "manifest.json"), indent=2))


## Azure ML managed online endpoint

`serving-render` chooses no-code MLflow deployment where Azure supports it and a custom
scoring entry point (the same service as the local contract test) otherwise. Deployment
checks the registered model's quality-gate, pipeline, scenario, mode and scope tags and
moves traffic last. `serving-invoke` adapts the factory request to the bundle's documented
contract (AutoML image column, AutoML forecast history). Endpoints bill while deployed:
delete them when finished.


In [ ]:
ONLINE_BUNDLE = INFERENCE / "azureml-online"
if RENDER_ONLINE:
    factory("serving-render", "--scenario", SCENARIO, "--runtime", EFFECTIVE_RUNTIME, "--kind", "online",
            "--model-name", MODEL_NAME, "--mode", MODE, "--output", ONLINE_BUNDLE)
if ONLINE_ENDPOINT is None and (ONLINE_BUNDLE / "manifest.json").exists():
    ONLINE_ENDPOINT = load_json(ONLINE_BUNDLE / "manifest.json")["endpoint_name"]
if DEPLOY_ONLINE:
    if not MODEL_ID:
        raise ValueError("Set MODEL_ID to the immutable registered model version")
    factory("serving-deploy", "--runtime", EFFECTIVE_RUNTIME, "--bundle", ONLINE_BUNDLE,
            "--model-id", MODEL_ID, "--execute")
if INVOKE_ONLINE:
    factory("serving-invoke", "--runtime", EFFECTIVE_RUNTIME, "--kind", "online", "--endpoint", ONLINE_ENDPOINT,
            "--request", REQUESTS / "online-request.json", "--bundle", ONLINE_BUNDLE, "--execute")
if DELETE_ONLINE:
    factory("serving-delete", "--runtime", EFFECTIVE_RUNTIME, "--kind", "online", "--endpoint", ONLINE_ENDPOINT,
            "--execute")


## Monitoring and clean-up

Join predictions with observed labels by `request_id` only after outcomes are known, then use
`python -m ml_model_factory monitor` with explicit reference/current windows (see the
model-factory guide). Predictions are never labels. Delete online endpoints (or Databricks serving endpoints) when the test is over.
